# WM 2026 – NLP Block: Conversational Agent

Dieser Block nimmt die WM-Prognose des ML-Modells und macht sie für den User zugänglich:  
- Der User fragt auf Deutsch nach einer Nation  
- GPT extrahiert die Nation als JSON  
- Das ML-Modell liefert die Prognose  
- GPT erklärt das Ergebnis in natürlicher Sprache  

**Pipeline:**  
`CV (Trikot → Nation)` → `ML (Nation → Prognose)` → **`NLP (Prognose → Erklärung)`**

Basiert auf demselben Ansatz wie Übung 3 (Wohnungspreise), angewendet auf WM-Vorhersagen.

---

## Setup

In [1]:
import json
import sys
import getpass
import joblib
import pandas as pd
import numpy as np
from openai import OpenAI
from pathlib import Path

# ML-Modell aus Block 1 laden
ML_MODEL_DIR = Path('../../Block1 ML/models')
model    = joblib.load(ML_MODEL_DIR / 'wm2026_model.pkl')
features = joblib.load(ML_MODEL_DIR / 'feature_names.pkl')

# FIFA-Daten für historische Stats
DATA_DIR = Path('../../data/FIFA')

# OpenAI Key sicher eingeben (nicht hardcoden!)
OPENAI_API_KEY = getpass.getpass("OpenAI API Key eingeben: ")
client = OpenAI(api_key=OPENAI_API_KEY)

TARGET_NATIONS = [
    'Argentina', 'Australia', 'Belgium', 'Brazil', 'Colombia',
    'Croatia', 'England', 'France', 'Germany', 'Japan',
    'Netherlands', 'Portugal', 'South Africa', 'Spain', 'Switzerland', 'USA'
]

print('Setup complete. Modell geladen:', type(model).__name__)

Setup complete. Modell geladen: GradientBoostingClassifier


## 1. Hilfsfunktionen (aus ML-Block übernommen)

In [2]:
# FIFA-Daten laden (für historische Stats in der Erklärung)
NUMERIC_COLS = ['Position', 'Games Played', 'Win', 'Draw', 'Loss',
                'Goals For', 'Goals Against', 'Goal Difference', 'Points']

years = [y for y in range(1930, 2023, 4) if y not in [1942, 1946]]
all_data = []
for year in years:
    fp = DATA_DIR / f'FIFA - {year}.csv'
    if fp.exists():
        df = pd.read_csv(fp)
        df['Year'] = year
        for col in NUMERIC_COLS:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        all_data.append(df)

df_hist = pd.concat(all_data, ignore_index=True)
df_hist['Team'] = df_hist['Team'].replace({'West Germany': 'Germany', 'United States': 'USA'})

stage_order = ['Group Stage', 'Round of 16', 'Quarter-final', 'Semi-final', 'Final', 'Champion']

def position_to_stage(pos, year):
    if pos == 1: return 'Champion'
    elif pos == 2: return 'Final'
    elif pos <= 4: return 'Semi-final'
    elif pos <= 8: return 'Quarter-final'
    elif year >= 1986 and pos <= 16: return 'Round of 16'
    else: return 'Group Stage'

df_hist['Stage'] = df_hist.apply(lambda r: position_to_stage(r['Position'], r['Year']), axis=1)
df_hist['Stage_Num'] = df_hist['Stage'].map({s: i for i, s in enumerate(stage_order)})

def get_nation_features(team):
    past = df_hist[df_hist['Team'] == team]
    if len(past) == 0: return None
    last3 = past.nlargest(3, 'Year')
    sorted_past = past.nlargest(2, 'Year')['Stage_Num'].values
    trend = float(sorted_past[0] - sorted_past[1]) if len(past) >= 2 else 0.0
    return {
        'n_participations': len(past),
        'n_titles':   int((past['Stage'] == 'Champion').sum()),
        'n_finals':   int(past['Stage'].isin(['Champion','Final']).sum()),
        'n_semis':    int(past['Stage'].isin(['Champion','Final','Semi-final']).sum()),
        'avg_stage_all':   past['Stage_Num'].mean(),
        'avg_stage_last3': last3['Stage_Num'].mean(),
        'last_stage':      past.loc[past['Year'].idxmax(), 'Stage_Num'],
        'win_rate':        past['Win'].sum() / max(past['Games Played'].sum(), 1),
        'goal_diff_pg':    past['Goal Difference'].sum() / max(past['Games Played'].sum(), 1),
        'goals_for_pg':    past['Goals For'].sum() / max(past['Games Played'].sum(), 1),
        'trend': trend,
    }

def predict_nation(team):
    feat = get_nation_features(team)
    if feat is None: return None
    X = pd.DataFrame([feat])[features].fillna(0)
    prediction = model.predict(X)[0]
    proba = dict(zip(model.classes_, model.predict_proba(X)[0]))
    return {
        'team': team,
        'prediction': prediction,
        'probabilities': proba,
        'n_titles': feat['n_titles'],
        'n_participations': feat['n_participations'],
        'last_stage': stage_order[int(feat['last_stage'])],
    }

# Test
print(predict_nation('Brazil'))

{'team': 'Brazil', 'prediction': 'Deep Run', 'probabilities': {'Deep Run': np.float64(0.9569653251506766), 'Early Exit': np.float64(0.021134553338416635), 'Final Stage': np.float64(0.021900121510906725)}, 'n_titles': 5, 'n_participations': 22, 'last_stage': 'Quarter-final'}


## 2. Prompt Design – Schritt 1: Nation extrahieren

Der User schreibt frei auf Deutsch. GPT extrahiert die gesuchte Nation als JSON.

### Iteration 1: Einfacher Prompt

In [3]:
def extract_nation_v1(user_text: str) -> dict:
    """Iteration 1: Einfacher Prompt ohne explizite Nationenliste."""
    system_prompt = (
        "Du bist ein Assistent der Fussball-WM 2026. "
        "Extrahiere die genannte Nation aus dem Text des Users und gib sie als JSON zurück. "
        'Antworte NUR mit einem JSON-Objekt: {"nation": "Nationenname auf Englisch"}'
    )
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_text},
        ],
        response_format={'type': 'json_object'},
    )
    return json.loads(response.choices[0].message.content)

# Tests
tests = [
    'Wie weit kommt die Schweiz bei der WM 2026?',
    'Was sagst du zu Brasilien?',
    'Ich glaube Deutschland wird Weltmeister!',
    'Tell me about the USA chances.',
]
print('=== Iteration 1 Tests ===')
for t in tests:
    result = extract_nation_v1(t)
    print(f'  Input:  {t}')
    print(f'  Output: {result}')
    print()

=== Iteration 1 Tests ===
  Input:  Wie weit kommt die Schweiz bei der WM 2026?
  Output: {'nation': 'Switzerland'}

  Input:  Was sagst du zu Brasilien?
  Output: {'nation': 'Brazil'}

  Input:  Ich glaube Deutschland wird Weltmeister!
  Output: {'nation': 'Germany'}

  Input:  Tell me about the USA chances.
  Output: {'nation': 'USA'}



### Iteration 2: Verbesserter Prompt mit Nationenliste & Fehlerbehandlung

In [4]:
def extract_nation_v2(user_text: str) -> dict:
    """Iteration 2: Prompt mit expliziter Nationenliste und null-Fallback."""
    nations_str = ', '.join(TARGET_NATIONS)
    system_prompt = (
        f"Du bist ein Assistent für die Fussball-WM 2026. "
        f"Folgende 16 Nationen nehmen in unserer App teil: {nations_str}.\n\n"
        "Extrahiere die vom User genannte Nation und mappe sie auf den exakten englischen Namen aus der Liste. "
        "Falls keine Nation erkennbar ist oder sie nicht in der Liste steht, gib null zurück.\n"
        'Antworte NUR mit JSON: {"nation": "Exakter Name" oder null}'
    )
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_text},
        ],
        response_format={'type': 'json_object'},
    )
    result = json.loads(response.choices[0].message.content)
    nation = result.get('nation')
    if nation and nation not in TARGET_NATIONS:
        raise ValueError(f"Nation '{nation}' nicht in der unterstützten Liste.")
    return result

# Dieselben Tests + Edge Cases
edge_cases = [
    'Wie läuft es für die Nati?',          # Schweizer Slang
    'Was ist mit den Oranje?',             # Spitzname Niederlande
    'Und Marokko?',                        # Nation nicht in Liste
    'Ich weiss nicht wen ich anfeuern soll.',  # Keine Nation
]

print('=== Iteration 2 Tests ===')
for t in tests + edge_cases:
    try:
        result = extract_nation_v2(t)
        print(f'  ✓  {t[:50]:50s} → {result}')
    except ValueError as e:
        print(f'  ✗  {t[:50]:50s} → Fehler: {e}')

=== Iteration 2 Tests ===
  ✓  Wie weit kommt die Schweiz bei der WM 2026?        → {'nation': 'Switzerland'}
  ✓  Was sagst du zu Brasilien?                         → {'nation': 'Brazil'}
  ✓  Ich glaube Deutschland wird Weltmeister!           → {'nation': 'Germany'}
  ✓  Tell me about the USA chances.                     → {'nation': 'USA'}
  ✓  Wie läuft es für die Nati?                         → {'nation': None}
  ✓  Was ist mit den Oranje?                            → {'nation': 'Netherlands'}
  ✓  Und Marokko?                                       → {'nation': None}
  ✓  Ich weiss nicht wen ich anfeuern soll.             → {'nation': None}


### Vergleich Iter 1 vs Iter 2

| Kriterium | Iteration 1 | Iteration 2 |
|---|---|---|
| Nationenliste im Prompt | Nein | Ja |
| Spitznamen-Mapping | Manchmal | Besser |
| Ungültige Nation | Kein Fallback | null + Fehler |
| Robustheit | Mittel | Hoch |

## 3. Prompt Design – Schritt 2: WM-Ergebnis erklären

### Iteration 1: Einfache Erklärung

In [5]:
def explain_prediction_v1(prediction_result: dict) -> str:
    """Iteration 1: Kurze direkte Erklärung."""
    system_prompt = (
        "Du bist ein Fussball-Experte. Erkläre die WM 2026 Prognose für eine Nation "
        "in 2-3 Sätzen auf Deutsch. Sei direkt und enthusiastisch."
    )
    user_prompt = f"Nation: {prediction_result['team']}\nPrognose: {prediction_result['prediction']}\nTitel: {prediction_result['n_titles']}\nLetzte WM: {prediction_result['last_stage']}"

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content

test_result = predict_nation('Brazil')
print('=== Iteration 1 Erklärung ===')
print(explain_prediction_v1(test_result))

=== Iteration 1 Erklärung ===
Brasilien hat das Potenzial für eine tiefgehende WM-Run 2026! Mit talentierten Spielern und einer starken Teamdynamik könnten sie ihren fünften Titel zurückholen. Nach dem Aus im Viertelfinale der letzten WM sind sie heiß darauf, das Turnier zu dominieren! Vamos, Brasilien!


### Iteration 2: Strukturierte Erklärung mit JSON + Wahrscheinlichkeiten

In [6]:
def explain_prediction_v2(prediction_result: dict) -> str:
    """Iteration 2: Strukturierte Erklärung mit Wahrscheinlichkeiten, gibt JSON zurück."""
    proba = prediction_result['probabilities']
    proba_str = ', '.join([f"{k}: {v:.0%}" for k, v in sorted(proba.items(), key=lambda x: -x[1])])

    system_prompt = (
        "Du bist ein Fussball-Experte und KI-Assistent für die WM 2026. "
        "Du erhältst eine ML-Modell-Prognose für eine Nation und erklärst sie dem User auf Deutsch. "
        "Deine Erklärung soll:\n"
        "- die Prognose klar nennen (Early Exit / Deep Run / Final Stage)\n"
        "- 1-2 historische Fakten der Nation einbeziehen\n"
        "- die Wahrscheinlichkeiten kurz erwähnen\n"
        "- eine Limitation des Modells nennen (nur historische Daten, kein aktueller Kader)\n"
        "Sei freundlich und fussballbegeistert. Max. 4 Sätze.\n\n"
        'Antworte NUR mit JSON: {"answer": "Deine Erklärung"}'
    )
    user_prompt = (
        f"Nation: {prediction_result['team']}\n"
        f"Prognose: {prediction_result['prediction']}\n"
        f"Wahrscheinlichkeiten: {proba_str}\n"
        f"WM-Titel: {prediction_result['n_titles']}\n"
        f"WM-Teilnahmen: {prediction_result['n_participations']}\n"
        f"Letzte WM: {prediction_result['last_stage']}"
    )

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        response_format={'type': 'json_object'},
    )
    result = json.loads(response.choices[0].message.content)
    return result['answer']

print('=== Iteration 2 Erklärung – Brazil ===')
print(explain_prediction_v2(predict_nation('Brazil')))
print()
print('=== Iteration 2 Erklärung – Switzerland ===')
print(explain_prediction_v2(predict_nation('Switzerland')))

=== Iteration 2 Erklärung – Brazil ===
Die Prognose für Brasilien lautet "Deep Run" mit einer Wahrscheinlichkeit von 96%, was darauf hindeutet, dass die Nation sehr wahrscheinlich bis ins Halbfinale vordringen wird. Brasilien ist der erfolgreichste Verein in der WM-Geschichte mit 5 Titeln und hat beeindruckende 22 Mal am Turnier teilgenommen. Allerdings zeigt das Modell nur die historischen Daten, ohne den aktuellen Kader zu berücksichtigen, was die Prognose beeinflussen könnte.

=== Iteration 2 Erklärung – Switzerland ===
Die Prognose für die Schweiz lautet: Deep Run mit einer Wahrscheinlichkeit von 54%. Die Schweizer Nationalmannschaft hat in ihrer Geschichte bereits 12 Mal an einer WM teilgenommen und zuletzt die Runde der letzten 16 erreicht, was zeigt, dass sie in großen Turnieren konkurrenzfähig sind. Es gibt auch noch Raum für Verbesserungen, denn sie haben bisher keinen WM-Titel gewonnen. Bitte beachten Sie, dass das Modell ausschließlich historische Daten nutzt und aktuelle Ka

## 4. Vollständige Pipeline testen

In [7]:
def run_wm_pipeline(user_text: str) -> tuple:
    """
    Vollständige NLP-Pipeline:
    User-Text → Nation extrahieren → ML-Prognose → Erklärung
    Returns: (extracted_json, prediction_result, explanation)
    """
    try:
        if not user_text.strip():
            return {}, {}, 'Bitte eine Frage eingeben.'

        # Schritt 1: Nation extrahieren
        extracted = extract_nation_v2(user_text)
        nation = extracted.get('nation')

        if not nation:
            return extracted, {}, 'Keine unterstützte Nation erkannt. Bitte frage nach einer der 16 WM-Nationen.'

        # Schritt 2: ML-Prognose
        prediction = predict_nation(nation)
        if not prediction:
            return extracted, {}, f'Keine historischen Daten für {nation}.'

        # Schritt 3: Erklärung generieren
        explanation = explain_prediction_v2(prediction)

        return extracted, prediction, explanation

    except ValueError as e:
        return {}, {}, f'Fehler: {e}'
    except Exception as e:
        return {}, {}, f'Unerwarteter Fehler: {e}'


# Pipeline testen
test_inputs = [
    'Wie weit kommt die Schweiz bei der WM 2026?',
    'Ich denke Argentinien wird Weltmeister! Was sagt dein Modell?',
    'Was ist mit Japan?',
]

for text in test_inputs:
    print(f'\n{'='*60}')
    print(f'INPUT: {text}')
    extracted, prediction, explanation = run_wm_pipeline(text)
    print(f'EXTRACTED JSON: {extracted}')
    if prediction:
        print(f'PREDICTION: {prediction["prediction"]} | Proba: {prediction["probabilities"]}')
    print(f'EXPLANATION: {explanation}')


INPUT: Wie weit kommt die Schweiz bei der WM 2026?
EXTRACTED JSON: {'nation': 'Switzerland'}
PREDICTION: Deep Run | Proba: {'Deep Run': np.float64(0.5356613377600479), 'Early Exit': np.float64(0.4042866050755519), 'Final Stage': np.float64(0.06005205716440025)}
EXPLANATION: Die Prognose für die Schweiz lautet "Deep Run" mit einer Wahrscheinlichkeit von 54%. Historisch hat die Schweiz an 12 Weltmeisterschaften teilgenommen und erreichte zuletzt das Achtelfinale. Es ist jedoch wichtig zu beachten, dass unser Modell nur auf historischen Daten basiert und den aktuellen Kader nicht berücksichtigt. Daher könnte die tatsächliche Leistung abweichen.

INPUT: Ich denke Argentinien wird Weltmeister! Was sagt dein Modell?
EXTRACTED JSON: {'nation': 'Argentina'}
PREDICTION: Deep Run | Proba: {'Deep Run': np.float64(0.6406312130798102), 'Early Exit': np.float64(0.03650852722097642), 'Final Stage': np.float64(0.32286025969921334)}
EXPLANATION: Die Prognose für Argentinien ist ein "Deep Run" mit eine

## 5. Zusammenfassung

### Prompt-Vergleich: Nation extrahieren

| Iteration | Prompt-Design | Stärken | Schwächen |
|---|---|---|---|
| 1 | Kein Kontext, keine Liste | Einfach | Kein Fallback, falsche Namen möglich |
| 2 | Mit Nationenliste + null-Fallback | Robust, Spitznamen erkannt | Prompt länger |

### Prompt-Vergleich: Erklärung generieren

| Iteration | Prompt-Design | Stärken | Schwächen |
|---|---|---|---|
| 1 | Frei, keine Struktur | Kreativ | Kein JSON, unkontrolliert |
| 2 | JSON-Output, Wahrscheinlichkeiten, Limitation | Konsistent, informativ | Leicht formeller |

**Finaler Ansatz:** Iteration 2 für beide Schritte  
**Integration:** `run_wm_pipeline()` wird direkt in `app.py` importiert